# MongoDB Handling — PAMAP2 Ingestion

After installing the MongoDB server in your machine, you can use this notebook for handling the initial processes with the database.

Specifically, in this step, we utilize Python's `pymongo` library to exploit its capabilities for MongoDB server interaction.

The wearable hardware that the project was originally built around is not available for this offering of the course, so you will work with the **PAMAP2 Physical Activity Monitoring** dataset (bundled in `data/PAMAP2_Dataset/`). Treat this notebook as the equivalent of the original "data collection" step: parse the raw `.dat` files, transform each (subject, activity) segment into a MongoDB document, and load it into your collection. The rest of the project (`aiot_project_*.ipynb`) reads from MongoDB, **not** from the raw files.

**Important Note: Be sure that the MongoDB server is up and running as a service in the background.**

For example, in macOS, to run MongoDB (i.e. the mongod process) as a service, run:

* `brew services start mongodb-community`

To stop a mongod running as a macOS service, use the following command as needed:

* `brew services stop mongodb-community`

To install MongoDB in your system, follow the instructions here:

* https://www.mongodb.com/docs/manual/administration/install-community/


**Note:** You can modify any of the processes below, however, you have to explain your thoughts.

In [ ]:
# import library for various processes with the OS
import os

## Load configuration

In [ ]:
# import library for yaml handling
import yaml

In [ ]:
config_path = os.path.join(os.getcwd(), "config.yml")

with open(config_path) as file:
    config = yaml.load(file, Loader=yaml.FullLoader)

## MongoDB database instantiation

The relevant information for the MongoDB client connection, the database name, and collection name is located in the configuration file.

```
# DB Connection with the uri (host)
client: "mongodb://localhost:27017/"

# db name
db: "aiot_course"

# db collection
col: "NAME YOUR COLLECTION"
```

In [ ]:
# import library for hanlding the MongoDB client
import pymongo
# import library for retrieving datetime
from datetime import datetime

### Create the database

To create a database in MongoDB, start by creating a MongoClient object, then specify a connection URL with the correct ip address and the name of the database you want to create.

MongoDB will create the database if it does not exist, and make a connection to it.

In [ ]:
client = pymongo.MongoClient(config["client"])

In [ ]:
db = client[config["db"]]

### Instantiate the collection

To create a collection in MongoDB, use the database object and specify the name of the collection you want to create.

MongoDB will create the collection if it does not exist.

In [ ]:
col = db[config["col"]]

Initially, no collection will be shown in MongoDB before you enter the first document!

## Create the data collection

You will populate the MongoDB collection from the PAMAP2 dataset under `data/PAMAP2_Dataset/`. The directory layout is fixed by the dataset and **must not be renamed or restructured**:

```
data/
└── PAMAP2_Dataset/
    ├── readme.pdf                 (canonical dataset description — read this first)
    ├── DataCollectionProtocol.pdf
    ├── DescriptionOfActivities.pdf
    ├── PerformedActivitiesSummary.pdf
    ├── subjectInformation.pdf
    ├── Protocol/                  (9 subjects, all required activities)
    │   ├── subject101.dat
    │   ├── ...
    │   └── subject109.dat
    └── Optional/                  (5 subjects, additional activities)
        ├── subject101.dat
        ├── ...
        └── subject109.dat
```

Each `.dat` file is a whitespace-separated text file with **54 columns** sampled at **100 Hz**. The full column layout (timestamp, activity_id, heart_rate, then 17 columns per IMU for hand/chest/ankle) is documented in `data/README.md` and in `PAMAP2_Dataset/readme.pdf`.

**Before inserting documents, your parsing code must:**

* Group the rows into **contiguous (subject, activity) segments** — a subject may perform the same activity in more than one block, and each block should become its own document, `activity_id == 0` (transient periods between activities).
* Ensure loaded data are in coordination with "Sensor Selection Strategy" in the root `README.md`.
* Ensure the working sampling rate is **100 Hz**. Downsample any channel that is not natively at 100 Hz.

In [ ]:
# import library for handling the .dat data and transformations
import pandas as pd
import numpy as np

Get dataset path:

In [ ]:
data_path = os.path.join(os.getcwd(), "data", "PAMAP2_Dataset")
print(data_path)

List all `.dat` files for each split (`Protocol`, `Optional`):

In [ ]:
splits = [s for s in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, s))]
print(splits)

In [ ]:
# print files in a split
split_path = os.path.join(data_path, "Protocol")
files_in_split = sorted(f for f in os.listdir(split_path)
                        if f.endswith(".dat") and os.path.isfile(os.path.join(split_path, f)))
print(files_in_split)

Each document in the MongoDB database should have the following schema (default configuration: hand/wrist IMU, accelerometer + gyroscope):

```json
{
  "_id": ObjectId("6984b3fa87abe7f4dff571aa"),
  "data": {
    "acc_x": ["array", "of", "values"],
    "acc_y": ["array", "of", "values"],
    "acc_z": ["array", "of", "values"],
    "gyr_x": ["array", "of", "values"],
    "gyr_y": ["array", "of", "values"],
    "gyr_z": ["array", "of", "values"]
  },
  "activity_id": 4,
  "activity_label": "walking",
  "subject": "101",
  "split": "Protocol",
  "imu_location": "hand",
  "sensor": "AccGyr",
  "sr": 100,
  "datetime": "MongoDB datetime object (it can be generated with the datetime.datetime.now() function)"
}
```

Field notes:

* `activity_id` is the integer label as it appears in the `.dat` file. `activity_label` is the human-readable name (see the activity table in the root `README.md`).
* `subject` is the file's subject identifier (e.g. `"101"` … `"109"`).
* `split` is `"Protocol"` or `"Optional"`, matching the source subdirectory.
* `imu_location` is `"hand"`, `"chest"`, or `"ankle"`. Use **one document per IMU location**; do not concatenate IMUs into a single document.
* `sensor` is `"Acc"`, `"Gyr"`, or `"AccGyr"`. If you later expand to the magnetometer, use `"AccGyrMag"` and add `mag_x`, `mag_y`, `mag_z` keys with the same axis convention.
* `sr` (sampling rate) **must be 100**. Downsample any channel that is not natively at 100 Hz before inserting.

If you are using only the gyroscope for an experiment, drop the `acc_*` keys and keep `gyr_x`, `gyr_y`, `gyr_z`. The same convention applies to the accelerometer-only case.

**Note: Be careful, the document is mandatory to have the aforementioned schema, in order to argue and proceed with the rest of the processes later on, in data engineering, plotting, etc.**

In [ ]:
from utils import df_rebase

## Provide the code to parse the PAMAP2 `.dat` files and upload the documents to MongoDB